In [1]:
# ── Cell 0: Setup (chạy đầu tiên) ──
import os

# Clone repo nếu chưa có
if not os.path.exists('/content/project'):
    !git clone https://github.com/TruongDuyLongPTIT/CTQW_PRO_METABOLITES_PRIORITIZING.git /content/project

# Add src vào Python path
import sys
sys.path.insert(0, '/content/project/src')

# Install dependencies nếu cần
!pip install -q torch scikit-learn networkx tqdm

Cloning into '/content/project'...
remote: Enumerating objects: 145, done.
remote: Counting objects: 100% (145/145), done.
remote: Compressing objects: 100% (122/122), done.
remote: Total 145 (delta 84), reused 56 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (145/145), 1021.12 KiB | 9.12 MiB/s, done.
Resolving deltas: 100% (84/84), done.


In [2]:
from pathlib import Path
from google.colab import drive

if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')
else:
    print("Drive đã mount sẵn.")

Mounted at /content/drive


In [10]:
!python /content/project/experiments/01_main_results.py

Traceback (most recent call last):
  File "/content/project/experiments/01_main_results.py", line 31, in <module>
    from methods import (
  File "/content/project/src/methods.py", line 11, in <module>
    from scipy.stats import rankdata as _rankdata
  File "/usr/local/lib/python3.12/dist-packages/scipy/stats/__init__.py", line 626, in <module>
    from ._stats_py import *
  File "/usr/local/lib/python3.12/dist-packages/scipy/stats/_stats_py.py", line 52, in <module>
    from . import distributions
  File "/usr/local/lib/python3.12/dist-packages/scipy/stats/distributions.py", line 11, in <module>
    from . import _discrete_distns
  File "<frozen importlib._bootstrap>", line 1360, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1331, in _find_and_load_unlocked
  File "<frozen importlib._bootstrap>", line 935, in _load_unlocked
  File "<frozen importlib._bootstrap_external>", line 995, in exec_module
  File "<frozen importlib._bootstrap_external>", line 1128, in get_cod

In [ ]:
!python /content/project/experiments/02_ablation_graph.py

Building graphs...
  G_pro: 2894 → clean: 2851 nodes
Building eval sets...
  Disease pathways: 20248
SMPDB files: 100% 48687/48687 [00:42<00:00, 1152.44it/s]
  Loaded 20244 pathways (errors=0)
  Dedup: 20244 → 3240 → 362 → 215
Eigendecomposition...

Running ablation...
PROF_o/HMDB+CTD: 100% 158/158 [09:20<00:00,  3.55s/it]
CTQW_o/HMDB+CTD: 100% 158/158 [10:24<00:00,  3.96s/it]
PROF_c/HMDB+CTD: 100% 158/158 [13:32<00:00,  5.14s/it]
CTQW_c/HMDB+CTD: 100% 158/158 [09:12<00:00,  3.50s/it]
  HMDB+CTD: 42.5 min
PROF_o/MarkerDB: 100% 21/21 [01:41<00:00,  4.83s/it]
CTQW_o/MarkerDB: 100% 21/21 [01:53<00:00,  5.39s/it]
PROF_c/MarkerDB: 100% 21/21 [02:24<00:00,  6.89s/it]
CTQW_c/MarkerDB: 100% 21/21 [01:40<00:00,  4.77s/it]
  MarkerDB: 7.7 min
PROF_o/SMPDB: 100% 153/153 [07:07<00:00,  2.79s/it]
CTQW_o/SMPDB: 100% 153/153 [07:51<00:00,  3.08s/it]
PROF_c/SMPDB: 100% 153/153 [10:13<00:00,  4.01s/it]
CTQW_c/SMPDB: 100% 153/153 [06:54<00:00,  2.71s/it]
  SMPDB: 32.1 min

ABLATION: Original vs Clean G_

In [11]:
!python /content/project/experiments/03_negative_results.py

Setup...
  Disease pathways: 20248
SMPDB files: 100% 48687/48687 [00:40<00:00, 1207.92it/s]
  Loaded 20244 pathways (errors=0)
  Dedup: 20244 → 3240 → 362 → 215
  HMDB+CTD: 158 diseases
  SMPDB:    153 diseases

EXP 1: Self-loop leakage analysis
  (A) Baseline:     CTQW-PRO, γ=0
  (B) Leaked:       H = A_pro + γ·diag(ALL mets)
  (C) Leakage-free: H = A_pro + γ·diag(seeds_only) per fold
Traceback (most recent call last):
  File "/content/project/experiments/03_negative_results.py", line 135, in <module>
    ev_c, vecs_c = np.linalg.eigh(H_c); del H_c
                   ^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/numpy/linalg/_linalg.py", line 1601, in eigh
    w, vt = gufunc(a, signature=signature)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt
^C


In [15]:
# ═══════════════════════════════════════════════════════════════════
# TEST: CTQW vs RWR — hub bias (degree correlation)
# Paste toàn bộ vào 1 cell Colab
# ═══════════════════════════════════════════════════════════════════
import sys
sys.path.insert(0, '/content/project/src')

import numpy as np
from scipy import stats

from config import CACHE_DIR, T_FIXED
from graph import (parse_recon3d, build_gcc, build_gpro,
                   build_hmdb_to_recon_initial, augment_hmdb_to_recon,
                   compute_eigendecomp)
from eval_sets import (parse_hmdb, build_hmdb_lookups,
                       build_cofactors_set, build_eval_set3)
from methods import make_profancy, make_ctqw_pro

# ── Setup ─────────────────────────────────────────────────────────
print('Setup...')
recon_data   = parse_recon3d()
G_cc, graph_nodes, N, node_idx, A_cc, degrees = build_gcc(recon_data)
met_info     = recon_data['met_info']
pathway_mets = recon_data['pathway_mets']

(G_pro, pro_nodes, N_PRO, idx_pro,
 A_pro, deg_pro, _pro_src, _pro_dst) = build_gpro(G_cc, node_idx, pathway_mets)

eigvals, eigvecs = compute_eigendecomp(A_pro, CACHE_DIR / 'gpro_eigdecomp.npz')

hmdb_data        = parse_hmdb()
hmdb_metabolites = hmdb_data['metabolites']
hmdb_lookups     = build_hmdb_lookups(hmdb_metabolites)
hmdb_to_recon    = build_hmdb_to_recon_initial(met_info, node_idx)
augment_hmdb_to_recon(
    hmdb_to_recon, met_info, node_idx,
    hmdb_lookups['ik_to_id'], hmdb_lookups['ikshort_to_id'],
    hmdb_lookups['name_to_id'], hmdb_lookups['name_aggr_to_id'])
COFACTORS = build_cofactors_set(hmdb_metabolites)
eval_set3 = build_eval_set3(hmdb_metabolites, hmdb_to_recon, node_idx)
print(f'  SMPDB: {len(eval_set3)} diseases')

# ── Build method functions ─────────────────────────────────────────
deg_safe     = np.where(A_pro.sum(1) > 0, A_pro.sum(1), 1.0)
P_pro        = A_pro / deg_safe[:, None]
run_profancy = make_profancy(P_pro, idx_pro, node_idx, N, N_PRO)
run_ctqw     = make_ctqw_pro(eigvals, eigvecs, idx_pro, N, N_PRO, _pro_src, _pro_dst)

# ── Map G_cc index → G_pro degree ─────────────────────────────────
# Scores tính trên G_pro → degree phải từ G_pro
gcc_to_pro_idx = {}
for nd, gcc_i in node_idx.items():
    if nd in idx_pro:
        gcc_to_pro_idx[gcc_i] = idx_pro[nd]

# ── Cofactor node indices (loại khỏi analysis) ────────────────────
cofactor_gcc_idx = set()
cof_names = set(COFACTORS)
for nd, i in node_idx.items():
    nd_base = nd.replace('_c','').replace('_m','').replace('_e','').replace('_x','')
    if nd in cof_names or nd_base in cof_names:
        cofactor_gcc_idx.add(i)
print(f'  Cofactor nodes excluded: {len(cofactor_gcc_idx)}')

# ── Thu thập scores từ 30 diseases ────────────────────────────────
N_DISEASES = 30
diseases_list = list(eval_set3.items())[:N_DISEASES]

rwr_scores_all  = []
ctqw_scores_all = []
degrees_all     = []

for disease, mets in diseases_list:
    valid = [m for m in mets if m in node_idx]
    if len(valid) < 3: continue

    # Dùng tất cả valid metabolites làm seed
    # Không cần LOO ở đây — chỉ cần xem score phân bổ thế nào
    seeds    = valid
    seed_set = {node_idx[s] for s in seeds}

    rwr_sc  = run_profancy(seeds)
    ctqw_sc = run_ctqw(seeds, [T_FIXED])[T_FIXED]

    for i in range(N):
        if i in seed_set:           # loại seed nodes
            continue
        if i in cofactor_gcc_idx:   # loại cofactor nodes
            continue
        if i not in gcc_to_pro_idx: # loại nodes không có trong G_pro
            continue
        rwr_scores_all.append(rwr_sc[i])
        ctqw_scores_all.append(ctqw_sc[i])
        degrees_all.append(deg_pro[gcc_to_pro_idx[i]])  # degree từ G_pro

rwr_arr  = np.array(rwr_scores_all)
ctqw_arr = np.array(ctqw_scores_all)
deg_arr  = np.array(degrees_all)

# ── Pearson correlation ────────────────────────────────────────────
r_rwr,  p_rwr  = stats.pearsonr(deg_arr, rwr_arr)
r_ctqw, p_ctqw = stats.pearsonr(deg_arr, ctqw_arr)

print('\n' + '='*55)
print('DEGREE CORRELATION — hub bias test')
print('='*55)
print(f'  n data points : {len(deg_arr):,}')
print(f'  RWR  r = {r_rwr:+.3f}  (p={p_rwr:.2e})')
print(f'  CTQW r = {r_ctqw:+.3f}  (p={p_ctqw:.2e})')
print(f'  Gap  = {r_rwr - r_ctqw:.3f}')
print(f"""
INTERPRETATION:
  RWR r >> CTQW r  → CTQW scores độc lập với degree hơn RWR
  → CTQW does not reward high-degree nodes disproportionately

  Nếu r_rwr > 0.3 và r_ctqw < 0.15:
  → ✅ Claim mạnh: "CTQW scores are largely decoupled from node degree
     (r = {r_ctqw:.3f}), whereas RWR scores correlate strongly with degree
     (r = {r_rwr:.3f}), indicating that CTQW does not reward
     high-connectivity hubs disproportionately."
""")
print('Done.')

ModuleNotFoundError: No module named 'config'

In [13]:
# ═══════════════════════════════════════════════════════════════════
# TEST: RWR vs CTQW vs NH-CTQW — hub bias (degree correlation)
# Paste toàn bộ vào 1 cell Colab
# ═══════════════════════════════════════════════════════════════════
import sys
sys.path.insert(0, '/content/project/src')

import numpy as np
from scipy import stats

from config import CACHE_DIR, T_FIXED, NH_GAMMA, RECON3D_COFACTORS
from graph import (parse_recon3d, build_gcc, build_gpro,
                   build_hmdb_to_recon_initial, augment_hmdb_to_recon,
                   compute_eigendecomp)
from eval_sets import (parse_hmdb, build_hmdb_lookups,
                       build_cofactors_set, build_eval_set3)
from methods import make_profancy, make_ctqw_pro, make_nh_pro

# ── Setup ─────────────────────────────────────────────────────────
print('Setup...')
recon_data   = parse_recon3d()
G_cc, graph_nodes, N, node_idx, A_cc, degrees = build_gcc(recon_data)
met_info     = recon_data['met_info']
pathway_mets = recon_data['pathway_mets']

(G_pro, pro_nodes, N_PRO, idx_pro,
 A_pro, deg_pro, _pro_src, _pro_dst) = build_gpro(G_cc, node_idx, pathway_mets)

eigvals, eigvecs = compute_eigendecomp(A_pro, CACHE_DIR / 'gpro_eigdecomp.npz')

hmdb_data        = parse_hmdb()
hmdb_metabolites = hmdb_data['metabolites']
hmdb_lookups     = build_hmdb_lookups(hmdb_metabolites)
hmdb_to_recon    = build_hmdb_to_recon_initial(met_info, node_idx)
augment_hmdb_to_recon(
    hmdb_to_recon, met_info, node_idx,
    hmdb_lookups['ik_to_id'], hmdb_lookups['ikshort_to_id'],
    hmdb_lookups['name_to_id'], hmdb_lookups['name_aggr_to_id'])
COFACTORS = build_cofactors_set(hmdb_metabolites)
eval_set3 = build_eval_set3(hmdb_metabolites, hmdb_to_recon, node_idx)
print(f'  SMPDB: {len(eval_set3)} diseases')

# ── Build method functions ─────────────────────────────────────────
print('Building methods (NH takes ~1 min)...')
deg_safe     = np.where(A_pro.sum(1) > 0, A_pro.sum(1), 1.0)
P_pro        = A_pro / deg_safe[:, None]

run_profancy = make_profancy(P_pro, idx_pro, node_idx, N, N_PRO)
run_ctqw     = make_ctqw_pro(eigvals, eigvecs, idx_pro, N, N_PRO, _pro_src, _pro_dst)
run_nh       = make_nh_pro(A_pro, idx_pro, N, N_PRO, _pro_src, _pro_dst,
                            RECON3D_COFACTORS, pro_nodes, NH_GAMMA, T_FIXED)
print('  Done.')

# ── Map G_cc index → G_pro degree ─────────────────────────────────
gcc_to_pro_idx = {}
for nd, gcc_i in node_idx.items():
    if nd in idx_pro:
        gcc_to_pro_idx[gcc_i] = idx_pro[nd]

# ── Cofactor node indices (loại khỏi analysis) ────────────────────
cofactor_gcc_idx = set()
cof_names = set(COFACTORS)
for nd, i in node_idx.items():
    nd_base = nd.replace('_c','').replace('_m','').replace('_e','').replace('_x','')
    if nd in cof_names or nd_base in cof_names:
        cofactor_gcc_idx.add(i)
print(f'  Cofactor nodes excluded: {len(cofactor_gcc_idx)}')

# ── Thu thập scores từ 30 diseases ────────────────────────────────
N_DISEASES = 30
diseases_list = list(eval_set3.items())[:N_DISEASES]

rwr_scores_all  = []
ctqw_scores_all = []
nh_scores_all   = []
degrees_all     = []

for disease, mets in diseases_list:
    valid = [m for m in mets if m in node_idx]
    if len(valid) < 3: continue

    seeds    = valid
    seed_set = {node_idx[s] for s in seeds}

    rwr_sc  = run_profancy(seeds)
    ctqw_sc = run_ctqw(seeds, [T_FIXED])[T_FIXED]
    nh_sc   = run_nh(seeds)

    for i in range(N):
        if i in seed_set:
            continue
        if i in cofactor_gcc_idx:
            continue
        if i not in gcc_to_pro_idx:
            continue
        rwr_scores_all.append(rwr_sc[i])
        ctqw_scores_all.append(ctqw_sc[i])
        nh_scores_all.append(nh_sc[i])
        degrees_all.append(deg_pro[gcc_to_pro_idx[i]])

rwr_arr  = np.array(rwr_scores_all)
ctqw_arr = np.array(ctqw_scores_all)
nh_arr   = np.array(nh_scores_all)
deg_arr  = np.array(degrees_all)

# ── Pearson correlation ────────────────────────────────────────────
r_rwr,  _ = stats.pearsonr(deg_arr, rwr_arr)
r_ctqw, _ = stats.pearsonr(deg_arr, ctqw_arr)
r_nh,   _ = stats.pearsonr(deg_arr, nh_arr)

print('\n' + '='*55)
print('DEGREE CORRELATION — hub bias test')
print('='*55)
print(f'  n data points : {len(deg_arr):,}')
print(f'  RWR      r = {r_rwr:+.3f}')
print(f'  CTQW     r = {r_ctqw:+.3f}')
print(f'  NH-CTQW  r = {r_nh:+.3f}')
print(f'\n  RWR → CTQW gap    : {r_rwr - r_ctqw:.3f}')
print(f'  CTQW → NH-CTQW gap: {r_ctqw - r_nh:.3f}')
print(f"""
INTERPRETATION:
  Nếu r_nh < r_ctqw < r_rwr:
  → ✅ NH-CTQW tiếp tục giảm hub bias mà CTQW chưa giải quyết hết
  → Narrative hoàn chỉnh: RWR → CTQW → NH-CTQW
     mỗi bước giảm hub bias thêm một tầng
""")
print('Done.')

ModuleNotFoundError: No module named 'config'

In [12]:
# ═══════════════════════════════════════════════════════════════════
# SCRIPT 1: Grid search t ∈ {0.05, 0.1, 0.2, 0.5}
# Chạy CTQW-PRO và PROFANCY trên HMDB+CTD để chọn t tối ưu
# Paste toàn bộ vào 1 cell Colab (~2-3 giờ full, ~30 phút sample)
# ═══════════════════════════════════════════════════════════════════
import sys
sys.path.insert(0, '/content/project/src')

import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

from config import CACHE_DIR, RESULTS_DIR, T_FIXED, RWR_ALPHA
from graph import (parse_recon3d, build_gcc, build_gpro,
                   build_hmdb_to_recon_initial, augment_hmdb_to_recon,
                   compute_eigendecomp)
from eval_sets import (parse_hmdb, build_hmdb_lookups,
                       build_cofactors_set, build_eval_set1)
from methods import make_ctqw_pro, make_profancy
from evaluation import run_loo_eval, wilcoxon_table

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Params ────────────────────────────────────────────────────────
T_GRID = [0.05, 0.1, 0.2, 0.5]   # grid search values
FULL_EVAL = True  # False = sample 40 diseases (~30 min)

# ── Setup ─────────────────────────────────────────────────────────
print('Setup...')
recon_data   = parse_recon3d()
G_cc, graph_nodes, N, node_idx, A_cc, degrees = build_gcc(recon_data)
met_info     = recon_data['met_info']
pathway_mets = recon_data['pathway_mets']

(G_pro, pro_nodes, N_PRO, idx_pro,
 A_pro, deg_pro, _pro_src, _pro_dst) = build_gpro(G_cc, node_idx, pathway_mets)

eigvals, eigvecs = compute_eigendecomp(A_pro, CACHE_DIR / 'gpro_eigdecomp.npz')

hmdb_data        = parse_hmdb()
hmdb_metabolites = hmdb_data['metabolites']
hmdb_lookups     = build_hmdb_lookups(hmdb_metabolites)
hmdb_to_recon    = build_hmdb_to_recon_initial(met_info, node_idx)
augment_hmdb_to_recon(
    hmdb_to_recon, met_info, node_idx,
    hmdb_lookups['ik_to_id'], hmdb_lookups['ikshort_to_id'],
    hmdb_lookups['name_to_id'], hmdb_lookups['name_aggr_to_id'])
COFACTORS = build_cofactors_set(hmdb_metabolites)
eval_set1, _ = build_eval_set1(
    hmdb_metabolites, hmdb_lookups, hmdb_to_recon, node_idx, COFACTORS)
print(f'  HMDB+CTD: {len(eval_set1)} diseases')

# ── Sample nếu cần ────────────────────────────────────────────────
if not FULL_EVAL:
    all_d = sorted(eval_set1.items(), key=lambda x: -len(x[1]))
    step = max(1, len(all_d) // 40)
    eval_set1 = dict(list(all_d)[i] for i in range(0, len(all_d), step))
    eval_set1 = dict(list(eval_set1.items())[:40])
    print(f'  Sample: {len(eval_set1)} diseases')

# ── PROFANCY baseline ──────────────────────────────────────────────
print('\nRunning PROFANCY (baseline)...')
deg_safe = np.where(A_pro.sum(1) > 0, A_pro.sum(1), 1.0)
P_pro    = A_pro / deg_safe[:, None]
run_prof = make_profancy(P_pro, idx_pro, node_idx, N, N_PRO)
df_prof  = run_loo_eval(eval_set1, run_prof, node_idx, N, label='PROFANCY')

# ── CTQW-PRO tại từng t ───────────────────────────────────────────
# make_ctqw_pro hỗ trợ batch: run(seeds, t_values=[...]) → {t: scores}
# → tính tất cả t values trong 1 LOO pass để tiết kiệm thời gian

print('\nRunning CTQW-PRO grid search over t...')
run_ctqw = make_ctqw_pro(eigvals, eigvecs, idx_pro, N, N_PRO, _pro_src, _pro_dst)

# Custom LOO cho batch t evaluation
rows_by_t = {t: [] for t in T_GRID}

for disease, mets in tqdm(eval_set1.items(), desc='CTQW t-grid'):
    valid = [m for m in mets if m in node_idx]
    if len(valid) < 3: continue
    for i, test_met in enumerate(valid):
        seeds    = [m for j, m in enumerate(valid) if j != i]
        seed_set = {node_idx[s] for s in seeds}
        test_idx = node_idx[test_met]
        try:
            # Batch: tính tất cả t values cùng lúc
            scores_dict = run_ctqw(seeds, t_values=T_GRID)
            for t in T_GRID:
                from evaluation import compute_metrics
                m = compute_metrics(scores_dict[t], test_idx, seed_set, _n=N)
                if m:
                    m['disease'] = disease
                    rows_by_t[t].append(m)
        except Exception:
            pass

# Aggregate per disease per t
results_by_t = {}
for t in T_GRID:
    if not rows_by_t[t]:
        continue
    tmp = pd.DataFrame(rows_by_t[t])
    results_by_t[t] = tmp.groupby('disease').mean(numeric_only=True).reset_index()

# ── Print results ─────────────────────────────────────────────────
print('\n' + '='*65)
print('GRID SEARCH t — CTQW-PRO on HMDB+CTD')
print('='*65)
print(f"{'t':>6}  {'AUC':>8} {'MRR':>8} {'R@5':>7} {'R@20':>7}  {'ΔMRR vs PROFANCY':>18}")
print('─'*65)

prof_mrr = df_prof['mrr'].mean() if df_prof is not None else 0

best_t = None; best_mrr = -1
for t in T_GRID:
    df = results_by_t.get(t)
    if df is None or df.empty: continue
    mrr = df['mrr'].mean()
    delta = mrr - prof_mrr
    marker = ' ← BEST' if mrr > best_mrr else ''
    print(f"  {t:>4}  {df['auc'].mean():>8.4f} {mrr:>8.4f} "
          f"{df['r@5'].mean():>7.4f} {df['r@20'].mean():>7.4f}  "
          f"{delta:>+10.4f}{marker}")
    if mrr > best_mrr:
        best_mrr = mrr; best_t = t

print(f'\nPROFANCY baseline: MRR={prof_mrr:.4f}')
print(f'Best t* = {best_t} (MRR={best_mrr:.4f})')
print(f'→ T_FIXED = {T_FIXED} {"✓ confirmed" if best_t == T_FIXED else "≠ check!"}')

# ── Save ──────────────────────────────────────────────────────────
rows_save = []
for t, df in results_by_t.items():
    if df is not None and not df.empty:
        tmp = df.copy(); tmp['t'] = t; rows_save.append(tmp)
if rows_save:
    out = RESULTS_DIR / 'grid_t_search.csv'
    pd.concat(rows_save).to_csv(out, index=False)
    print(f'Saved: {out}')
print('Done.')

ModuleNotFoundError: No module named 'config'

In [5]:
# ═══════════════════════════════════════════════════════════════════
# SCRIPT 2b: γ grid extended {50, 60, 80}
# Chạy sau grid_gamma_cell.py để tìm plateau
# Paste vào 1 cell Colab (~20 phút)
# ═══════════════════════════════════════════════════════════════════
import sys, os
for _p in ['/content/project/src',
           '/content/drive/MyDrive/CTQW for metabolites/src']:
    if os.path.isdir(_p) and _p not in sys.path:
        sys.path.insert(0, _p)

import numpy as np
import pandas as pd
import time

from config import CACHE_DIR, RESULTS_DIR, T_FIXED, NH_GAMMA, RECON3D_COFACTORS
from graph import (parse_recon3d, build_gcc, build_gpro,
                   build_hmdb_to_recon_initial, augment_hmdb_to_recon,
                   compute_eigendecomp)
from eval_sets import (parse_hmdb, build_hmdb_lookups,
                       build_cofactors_set, build_eval_set1, build_eval_set3)
from methods import make_nh_pro, make_ctqw_pro
from evaluation import run_loo_eval

# ── Params ────────────────────────────────────────────────────────
GAMMA_EXT = [150, 349]

# ── Setup ─────────────────────────────────────────────────────────
print('Setup...')
recon_data   = parse_recon3d()
G_cc, graph_nodes, N, node_idx, A_cc, degrees = build_gcc(recon_data)
met_info     = recon_data['met_info']
pathway_mets = recon_data['pathway_mets']

(G_pro, pro_nodes, N_PRO, idx_pro,
 A_pro, deg_pro, _pro_src, _pro_dst) = build_gpro(G_cc, node_idx, pathway_mets)

eigvals, eigvecs = compute_eigendecomp(A_pro, CACHE_DIR / 'gpro_eigdecomp.npz')

hmdb_data        = parse_hmdb()
hmdb_metabolites = hmdb_data['metabolites']
hmdb_lookups     = build_hmdb_lookups(hmdb_metabolites)
hmdb_to_recon    = build_hmdb_to_recon_initial(met_info, node_idx)
augment_hmdb_to_recon(
    hmdb_to_recon, met_info, node_idx,
    hmdb_lookups['ik_to_id'], hmdb_lookups['ikshort_to_id'],
    hmdb_lookups['name_to_id'], hmdb_lookups['name_aggr_to_id'])
COFACTORS = build_cofactors_set(hmdb_metabolites)
eval_set1, _ = build_eval_set1(
    hmdb_metabolites, hmdb_lookups, hmdb_to_recon, node_idx, COFACTORS)
eval_set3     = build_eval_set3(hmdb_metabolites, hmdb_to_recon, node_idx)
print(f'  HMDB+CTD: {len(eval_set1)}, SMPDB: {len(eval_set3)}')

# ── Load kết quả cũ từ grid {10,15,22,30,40} ──────────────────────
prev_file = RESULTS_DIR / 'grid_gamma_search.csv'
prev_data = {}
if prev_file.exists():
    df_prev = pd.read_csv(prev_file)
    for gamma in [10,15,22,30,40]:
        for label in ['HMDB+CTD','SMPDB']:
            sub = df_prev[(df_prev['gamma']==gamma) & (df_prev['dataset']==label)]
            if not sub.empty:
                prev_data[(gamma,label)] = sub
    print(f'  Loaded {len(prev_data)} prev results')

# CTQW-PRO baseline
ctqw_fn = lambda seeds: make_ctqw_pro(
    eigvals, eigvecs, idx_pro, N, N_PRO, _pro_src, _pro_dst
)(seeds, [T_FIXED])[T_FIXED]

# Compute baselines (hoặc load từ file nếu có)
base_file = RESULTS_DIR / 'grid_gamma_baseline.csv'
if base_file.exists():
    df_base_all = pd.read_csv(base_file)
    baselines = {}
    for label in ['HMDB+CTD','SMPDB']:
        sub = df_base_all[df_base_all['dataset']==label]
        baselines[label] = sub if not sub.empty else None
    print('  Loaded baselines from file')
else:
    print('  Computing baselines...')
    run_ctqw = make_ctqw_pro(eigvals, eigvecs, idx_pro, N, N_PRO, _pro_src, _pro_dst)
    ctqw_fn2 = lambda seeds: run_ctqw(seeds, [T_FIXED])[T_FIXED]
    baselines = {}
    for label, dset in [('HMDB+CTD',eval_set1),('SMPDB',eval_set3)]:
        baselines[label] = run_loo_eval(dset, ctqw_fn2, node_idx, N,
                                         label=f'CTQW/{label}')

# ── Run extended γ ────────────────────────────────────────────────
print('\nRunning NH extended γ grid...')
new_results = {}

for gamma in GAMMA_EXT:
    t0 = time.time()
    print(f'\n  γ={gamma}...', end=' ', flush=True)
    run_nh = make_nh_pro(
        A_pro, idx_pro, N, N_PRO, _pro_src, _pro_dst,
        RECON3D_COFACTORS, pro_nodes, gamma, T_FIXED)
    print(f'{time.time()-t0:.1f}s')
    for label, dset in [('HMDB+CTD',eval_set1),('SMPDB',eval_set3)]:
        df = run_loo_eval(dset, run_nh, node_idx, N, label=f'NH(γ={gamma})/{label}')
        new_results[(gamma,label)] = df

# ── Full picture ──────────────────────────────────────────────────
all_gammas = [10,15,22,30,40,50,60,80]
all_results = {**prev_data, **new_results}

print('\n' + '='*65)
print('COMPLETE γ GRID — NH-CTQW-PRO')
print('='*65)

for label in ['HMDB+CTD','SMPDB']:
    base = baselines.get(label)
    base_mrr = base['mrr'].mean() if base is not None and not base.empty else 0
    print(f'\n{label} (CTQW-PRO MRR={base_mrr:.4f}):')
    print(f"  {'γ':>5}  {'MRR':>8}  {'ΔMRR':>8}")
    print('  ' + '─'*28)
    best_mrr = -1
    for g in all_gammas:
        df = all_results.get((g, label))
        if df is None: continue
        mrr = df['mrr'].mean() if hasattr(df,'mrr') else df['mrr'].mean()
        delta = mrr - base_mrr
        marker = ' ← best' if mrr > best_mrr else ''
        print(f"  {g:>5}  {mrr:>8.4f}  {delta:>+8.4f}{marker}")
        if mrr > best_mrr: best_mrr = mrr

# ── Save extended ─────────────────────────────────────────────────
save_rows = []
for (gamma,label), df in new_results.items():
    if df is not None and not df.empty:
        tmp = df.copy(); tmp['gamma']=gamma; tmp['dataset']=label
        save_rows.append(tmp)
if save_rows:
    df_new = pd.concat(save_rows)
    if prev_file.exists():
        df_old = pd.read_csv(prev_file)
        pd.concat([df_old, df_new]).to_csv(prev_file, index=False)
    else:
        df_new.to_csv(prev_file, index=False)
    print(f'\nAppended to {prev_file}')
print('Done.')

Setup...
  Disease pathways: 20248


SMPDB files:   0%|          | 0/48687 [00:00<?, ?it/s]

  Loaded 20244 pathways (errors=0)
  Dedup: 20244 → 3240 → 362 → 215
  HMDB+CTD: 158, SMPDB: 153
  Loaded 10 prev results
  Loaded baselines from file

Running NH extended γ grid...

  γ=150... 91.7s


NH(γ=150)/HMDB+CTD:   0%|          | 0/158 [00:00<?, ?it/s]

NH(γ=150)/SMPDB:   0%|          | 0/153 [00:00<?, ?it/s]


  γ=349... 92.7s


NH(γ=349)/HMDB+CTD:   0%|          | 0/158 [00:00<?, ?it/s]

NH(γ=349)/SMPDB:   0%|          | 0/153 [00:00<?, ?it/s]


COMPLETE γ GRID — NH-CTQW-PRO

HMDB+CTD (CTQW-PRO MRR=0.0598):
      γ       MRR      ΔMRR
  ────────────────────────────
     10    0.0644   +0.0047 ← best
     15    0.0655   +0.0058 ← best
     22    0.0658   +0.0060 ← best
     30    0.0661   +0.0063 ← best
     40    0.0659   +0.0061

SMPDB (CTQW-PRO MRR=0.2172):
      γ       MRR      ΔMRR
  ────────────────────────────
     10    0.2564   +0.0392 ← best
     15    0.2584   +0.0412 ← best
     22    0.2618   +0.0446 ← best
     30    0.2651   +0.0479 ← best
     40    0.2674   +0.0502 ← best

Appended to /content/drive/MyDrive/CTQW for metabolites/results/grid_gamma_search.csv
Done.


In [6]:
import pandas as pd
from config import RESULTS_DIR

df = pd.read_csv(RESULTS_DIR / 'grid_gamma_search.csv')

print("SMPDB MRR theo γ:")
smpdb = df[df['dataset']=='SMPDB'].groupby('gamma')['mrr'].mean().sort_index()
print(smpdb.to_string())

print("\nHMDB+CTD MRR theo γ:")
hmdb = df[df['dataset']=='HMDB+CTD'].groupby('gamma')['mrr'].mean().sort_index()
print(hmdb.to_string())

SMPDB MRR theo γ:
gamma
10     0.256405
15     0.258404
22     0.261797
30     0.265123
40     0.267435
50     0.267303
60     0.268083
80     0.269119
150    0.269759
349    0.269051

HMDB+CTD MRR theo γ:
gamma
10     0.064429
15     0.065540
22     0.065766
30     0.066084
40     0.065919
50     0.065422
60     0.065102
80     0.064484
150    0.063880
349    0.063195


In [9]:
# ═══════════════════════════════════════════════════════════════════
# SCRIPT 3 (ULTRAFAST): Driven grid - vectorized over all alphas
# Trick: với fixed psi_seed và fixed e^{-iAt}, precompute
#        tất cả walked states, dùng numpy broadcasting
# Ước tính: ~3-5 phút cho 50 diseases
# ═══════════════════════════════════════════════════════════════════
import sys, os
for _p in ['/content/project/src',
           '/content/drive/MyDrive/CTQW for metabolites/src']:
    if os.path.isdir(_p) and _p not in sys.path:
        sys.path.insert(0, _p)

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from config import CACHE_DIR, RESULTS_DIR, T_FIXED, DRIVEN_N_STEPS, DRIVEN_ALPHA
from graph import (parse_recon3d, build_gcc, build_gpro,
                   build_hmdb_to_recon_initial, augment_hmdb_to_recon,
                   compute_eigendecomp)
from eval_sets import (parse_hmdb, build_hmdb_lookups,
                       build_cofactors_set, build_eval_set3)
from evaluation import compute_metrics

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

N_STEPS_GRID = [1, 2, 3, 5, 10, 15, 20]
ALPHA_GRID   = [0.1, 0.2, 0.3, 0.5, 0.8]
N_DISEASES   = 50

# ── Setup ─────────────────────────────────────────────────────────
print('Setup...')
recon_data   = parse_recon3d()
G_cc, graph_nodes, N, node_idx, A_cc, degrees = build_gcc(recon_data)
met_info     = recon_data['met_info']
pathway_mets = recon_data['pathway_mets']
(G_pro, pro_nodes, N_PRO, idx_pro,
 A_pro, deg_pro, _pro_src, _pro_dst) = build_gpro(G_cc, node_idx, pathway_mets)
eigvals, eigvecs = compute_eigendecomp(A_pro, CACHE_DIR / 'gpro_eigdecomp.npz')

hmdb_data        = parse_hmdb()
hmdb_metabolites = hmdb_data['metabolites']
hmdb_lookups     = build_hmdb_lookups(hmdb_metabolites)
hmdb_to_recon    = build_hmdb_to_recon_initial(met_info, node_idx)
augment_hmdb_to_recon(
    hmdb_to_recon, met_info, node_idx,
    hmdb_lookups['ik_to_id'], hmdb_lookups['ikshort_to_id'],
    hmdb_lookups['name_to_id'], hmdb_lookups['name_aggr_to_id'])
COFACTORS = build_cofactors_set(hmdb_metabolites)
eval_set3 = build_eval_set3(hmdb_metabolites, hmdb_to_recon, node_idx)

all_d = sorted(eval_set3.items(), key=lambda x: -len(x[1]))
step  = max(1, len(all_d) // N_DISEASES)
sample = dict(list(all_d)[i] for i in range(0, len(all_d), step))
sample = dict(list(sample.items())[:N_DISEASES])
print(f'  {len(sample)} diseases, {len(N_STEPS_GRID)}×{len(ALPHA_GRID)}={len(N_STEPS_GRID)*len(ALPHA_GRID)} combos')

phases    = np.exp(-1j * eigvals * T_FIXED)   # (N_PRO,)
Vt        = eigvecs.conj().T                   # (N_PRO, N_PRO) — precompute once
alpha_arr = np.array(ALPHA_GRID)               # (5,)
MAX_STEPS = max(N_STEPS_GRID)                  # = 20
steps_set = set(N_STEPS_GRID)

def gcc_scores_from_probs(probs):
    sc = np.zeros(N); sc[_pro_dst] = probs[_pro_src]; return sc

# ── Ultra-fast core: vectorized over all 5 alphas simultaneously ──
def compute_all_combos(valid_pro_idx):
    """
    Returns: dict {(n_steps, alpha): probs (N_PRO,)}

    KEY INSIGHT: run 5 alpha tracks in parallel using shape (5, N_PRO)
    Each row = one alpha's psi state
    Matrix op: eigvecs @ (phases * (Vt @ psi.T)) vectorized over all alphas
    """
    # psi_seed shape: (N_PRO,) → broadcast to (5, N_PRO)
    ps = np.zeros(N_PRO, dtype=complex)
    ps[valid_pro_idx] = 1.0 / np.sqrt(len(valid_pro_idx))

    # psi_mat[a] = current psi for alpha_arr[a], shape (5, N_PRO)
    psi_mat = np.tile(ps, (len(alpha_arr), 1))  # (5, N_PRO)

    results = {}

    for k in range(1, MAX_STEPS + 1):
        # Vectorized evolution: walked[a] = e^{-iAt} @ psi_mat[a]
        # Vt @ psi_mat.T = (N_PRO, 5), then phases * (...) = (N_PRO, 5)
        # eigvecs @ (...) = (N_PRO, 5) → transpose → (5, N_PRO)
        coef_mat  = Vt @ psi_mat.T              # (N_PRO, 5)
        walked_mat = (eigvecs @ (phases[:, None] * coef_mat)).T  # (5, N_PRO)

        # Mix: psi = (1-alpha)*walked + alpha*psi_seed
        # alpha_arr shape (5,) → (5,1) for broadcasting
        psi_mat = ((1 - alpha_arr[:, None]) * walked_mat
                   + alpha_arr[:, None] * ps[None, :])

        # Normalize each row
        norms = np.linalg.norm(psi_mat, axis=1, keepdims=True)  # (5,1)
        psi_mat = np.where(norms > 1e-9, psi_mat / norms, psi_mat)

        # Save results at requested n_steps
        if k in steps_set:
            probs_mat = np.abs(psi_mat)**2  # (5, N_PRO)
            for ai, alpha in enumerate(ALPHA_GRID):
                results[(k, alpha)] = probs_mat[ai]

    return results

# ── LOO evaluation ────────────────────────────────────────────────
print('Running LOO...')

# Accumulators: combo → list of per-disease mean metrics
combo_acc = {(n,a): [] for n in N_STEPS_GRID for a in ALPHA_GRID}
base_acc  = []

for disease, mets in tqdm(sample.items(), desc='LOO', leave=True):
    valid = [m for m in mets if m in node_idx]
    if len(valid) < 3: continue

    combo_folds = {(n,a): [] for n in N_STEPS_GRID for a in ALPHA_GRID}
    base_folds  = []

    for i, test_met in enumerate(valid):
        seeds    = [m for j, m in enumerate(valid) if j != i]
        seed_set = {node_idx[s] for s in seeds}
        test_idx = node_idx[test_met]
        valid_pro = [idx_pro[s] for s in seeds if s in idx_pro]
        if not valid_pro: continue

        # CTQW baseline
        ps0 = np.zeros(N_PRO, dtype=complex)
        ps0[valid_pro] = 1.0 / np.sqrt(len(valid_pro))
        base_probs = np.abs(eigvecs @ (phases * (Vt @ ps0)))**2
        m = compute_metrics(gcc_scores_from_probs(base_probs),
                            test_idx, seed_set, _n=N)
        if m: base_folds.append(m)

        # All 35 combos — vectorized
        all_probs = compute_all_combos(valid_pro)
        for (n_steps, alpha), probs in all_probs.items():
            m = compute_metrics(gcc_scores_from_probs(probs),
                                test_idx, seed_set, _n=N)
            if m: combo_folds[(n_steps, alpha)].append(m)

    # Aggregate folds
    if base_folds:
        row = {k: float(np.mean([r[k] for r in base_folds]))
               for k in ['auc','mrr','r@5','r@20']}
        row['disease'] = disease
        base_acc.append(row)

    for key, folds in combo_folds.items():
        if folds:
            row = {k: float(np.mean([r[k] for r in folds]))
                   for k in ['auc','mrr','r@5','r@20']}
            row['disease'] = disease
            combo_acc[key].append(row)

# ── Results ───────────────────────────────────────────────────────
df_base  = pd.DataFrame(base_acc) if base_acc else None
base_mrr = df_base['mrr'].mean() if df_base is not None else 0
dfs      = {k: pd.DataFrame(v) if v else None for k, v in combo_acc.items()}

print(f'\nCTQW-PRO baseline MRR = {base_mrr:.4f}')
print()
header = f"{'n_steps':>8} |" + ''.join(f"  α={a}" for a in ALPHA_GRID)
print(header); print('─'*len(header))

best_combo = None; best_mrr_g = -1; grid_vals = {}
for n_steps in N_STEPS_GRID:
    row_str = f"{n_steps:>8} |"
    for alpha in ALPHA_GRID:
        df  = dfs.get((n_steps, alpha))
        mrr = df['mrr'].mean() if df is not None and not df.empty else float('nan')
        grid_vals[(n_steps, alpha)] = mrr
        marker = '*' if mrr > best_mrr_g else ' '
        row_str += f"  {mrr:.3f}{marker}"
        if mrr > best_mrr_g:
            best_mrr_g = mrr; best_combo = (n_steps, alpha)
    print(row_str)

paper_mrr = grid_vals.get((DRIVEN_N_STEPS, DRIVEN_ALPHA), float('nan'))
print(f'\n* Best:  n_steps={best_combo[0]}, α={best_combo[1]} '
      f'(MRR={best_mrr_g:.4f}, Δ={best_mrr_g-base_mrr:+.4f})')
print(f'  Paper: n_steps={DRIVEN_N_STEPS}, α={DRIVEN_ALPHA} '
      f'(MRR={paper_mrr:.4f}, Δ={paper_mrr-base_mrr:+.4f})')
diff = paper_mrr - best_mrr_g
if abs(diff) < 0.003:
    print(f'  → ✅ Paper params near-optimal (gap={diff:+.4f})')
else:
    print(f'  → ⚠️  Gap = {diff:+.4f}, consider updating params')

# ── Save ──────────────────────────────────────────────────────────
rows_save = []
for (n_steps, alpha), df in dfs.items():
    if df is not None and not df.empty:
        tmp = df.copy(); tmp['n_steps'] = n_steps; tmp['alpha'] = alpha
        rows_save.append(tmp)
if rows_save:
    out = RESULTS_DIR / 'grid_driven_search.csv'
    pd.concat(rows_save).to_csv(out, index=False)
    print(f'Saved: {out}')
if df_base is not None:
    df_base.to_csv(RESULTS_DIR / 'grid_driven_baseline.csv', index=False)
print('Done.')

Setup...
  Disease pathways: 20248


SMPDB files:   0%|          | 0/48687 [00:00<?, ?it/s]

  Loaded 20244 pathways (errors=0)
  Dedup: 20244 → 3240 → 362 → 215
  50 diseases, 7×5=35 combos
Running LOO...


LOO:   0%|          | 0/50 [00:00<?, ?it/s]


CTQW-PRO baseline MRR = 0.2163

 n_steps |  α=0.1  α=0.2  α=0.3  α=0.5  α=0.8
─────────────────────────────────────────────
       1 |  0.216*  0.216   0.216   0.216   0.216 
       2 |  0.256*  0.257*  0.255   0.257*  0.237 
       3 |  0.190   0.207   0.228   0.248   0.239 
       5 |  0.177   0.182   0.191   0.231   0.239 
      10 |  0.102   0.151   0.178   0.227   0.239 
      15 |  0.087   0.122   0.170   0.226   0.239 
      20 |  0.082   0.120   0.168   0.226   0.239 

* Best:  n_steps=2, α=0.5 (MRR=0.2569, Δ=+0.0406)
  Paper: n_steps=2, α=0.5 (MRR=0.2569, Δ=+0.0406)
  → ✅ Paper params near-optimal (gap=+0.0000)
Saved: /content/drive/MyDrive/CTQW for metabolites/results/grid_driven_search.csv
Done.
